In [0]:
dbutils.widgets.text("catalog", "dbr_dev")
dbutils.widgets.text("schema", "brazilian_ecommerce_bronze")
dbutils.widgets.text("table_name", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
table_name = dbutils.widgets.get("table_name")

In [0]:
volume_name = "landing"

source_path = f"/Volumes/{catalog}/{schema}/{volume_name}/{table_name}/"
checkpoint_path = f"/Volumes/{catalog}/{schema}/{volume_name}/checkpoints/{table_name}/"

In [0]:
df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", checkpoint_path)
    .option("header", "true")
    .load(source_path)
)

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    df
    .withColumn("source_file", F.col("_metadata.file_name"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

(
    df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .option("mergeSchema", "true")
    .toTable(f"{catalog}.{schema}.brz_{table_name}")
)